In [6]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator, Pauli
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 
import numpy.linalg as la

random_backend = BasicSimulator()

def quantum_random_bit():
    q = QuantumCircuit(1, 1)
    q.h(0)
    q.measure(0, 0)
    compiled = transpile(q, random_backend)
    result = random_backend.run(compiled, shots=1).result()
    counts = result.get_counts(compiled)
    return 1 if counts.get('1', 0) else 0

def quantum_random_int(max_exclusive):
    bits_needed = math.ceil(math.log2(max_exclusive))
    while True:
        value = 0
        for _ in range(bits_needed):
            value = (value << 1) | quantum_random_bit()
        if value < max_exclusive:
            return value

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.
# The attack destroys entanglement, reducing |S| to below 2.
# Not all of the shifted key bits agree after Bob inverts his bits, showing that the protocol is not secure. (see results at the bottom)

In [ ]:
X = Operator(Pauli('X'))
Z = Operator(Pauli('Z'))

W = Operator(X.data + Z.data) / np.sqrt(2)
V = Operator(X.data - Z.data) / np.sqrt(2)

operators = { 
  "A": [X, W, Z], 
  "B": [W, Z, V]
}

psi_minus = Statevector([0, 1/np.sqrt(2), -1/np.sqrt(2), 0])

U_A = []
U_B = []

for op in operators["A"]:
    values, vectors = la.eigh(op.data)
    U_A.append(vectors.conj().T)
    
for op in operators["B"]:
    values, vectors = la.eigh(op.data)
    U_B.append(vectors.conj().T)

def run_round():
    """Simulate one E91 round: returns (i, j, a, b)."""
    i = quantum_random_int(3)  # Alice choice index
    j = quantum_random_int(3)  # Bob choice index
    
    # attacker logic
    eve_basis = quantum_random_int(3)

    attacked_sv = psi_minus.evolve(U_B[eve_basis], [1])
    _, attacked_sv = attacked_sv.measure([1])
    attacked_sv = attacked_sv.evolve(U_B[eve_basis].conj().T, [1])
    

    sv = attacked_sv.evolve(U_A[i], [0]).evolve(U_B[j], [1])
    bitstring = next(iter(sv.sample_counts(shots=1).keys()))
    a, b = int(bitstring[0]), int(bitstring[1])  # alice and bob's measurement results
    return i, j, a, b


In [ ]:
N_rounds = 50_000 # this takes quite a while to run, so could be reduced for testing
key_bits = []
alice_bases = []
bob_bases = []
alice_bits = []
bob_bits = []
alice_bob_agreements = 0
sifted_rounds = 0

E_sums = {(0,0):0, (0,2):0, (2,0):0, (2,2):0}
E_counts = {k:0 for k in E_sums}

# only show the first few values
def preview(values, limit=20):
    shown = ", ".join(str(v) for v in values[:limit])
    if len(values) > limit:
        shown += ", ..."
    return f"[{shown}]"

for _ in range(N_rounds):
    i, j, a, b = run_round()
    alice_bases.append(i)
    bob_bases.append(j)
    alice_bits.append(a)
    bob_bits.append(b)
    
    if (i, j) in {(1, 0), (2, 1)}:  # sifted key round
        key_bits.append(a)
        sifted_rounds += 1

        bob_corrected = 1 - b
        if a == bob_corrected: alice_bob_agreements += 1

    # Accumulate correlators for S
    if (i, j) in E_sums:
        va = 1 if a == 0 else -1
        vb = 1 if b == 0 else -1
        E_sums[(i, j)] += va * vb
        E_counts[(i, j)] += 1

E = {k: E_sums[k]/E_counts[k] for k in E_sums if E_counts[k] != 0}
S = E[(0,0)] + E[(0,2)] + E[(2,0)] - E[(2,2)]

print(f"S estimate: {S:.3f}, |S|={abs(S):.3f}")
print(f"Key length (with attacker): {len(key_bits)} ({len(key_bits)/N_rounds:.2%} of {N_rounds} rounds)")
print(f"Alice bases: {preview(alice_bases)}")
print(f"Bob bases:   {preview(bob_bases)}")
print(f"Alice bits:  {preview(alice_bits)}")
print(f"Bob bits:    {preview(bob_bits)}")
print(f"Shared key (length {len(key_bits)}): {preview(key_bits)}")
print(f"Sifted key agreement rate: {alice_bob_agreements/sifted_rounds:.2%}")



S estimate: -1.395, |S|=1.395
Key length (with attacker): 11106 (22.21% of 50000 rounds)
Alice bases: [0, 1, 1, 2, 2, 1, 2, 0, 2, 1, 2, 1, 2, 2, 1, 0, 0, 1, 0, 0, ...]
Bob bases:   [2, 2, 1, 0, 1, 0, 2, 0, 0, 0, 1, 2, 1, 2, 2, 1, 2, 1, 1, 0, ...]
Alice bits:  [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, ...]
Bob bits:    [1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, ...]
Shared key (length 11106): [0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, ...]
Sifted key agreement rate: 78.93%
